In [ ]:
# === Mount Google Drive ===
from google.colab import drive
drive.mount('/content/drive')

# === Install necessary packages (only run once) ===
!pip install -q mne plotly colorama pyphen

# === Imports ===
import os
import gc
import traceback

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import librosa
import pyphen
import mne
import plotly.graph_objects as go
from IPython.display import display
from colorama import Fore, Style, init

# === Colorama init (for colored terminal printouts) ===
init(autoreset=True)


# Hypothesis:
Null Hypothesis (H₀):
ERP and acoustic features (e.g., speech rate, inter-word interval, MFCCs, F0, etc.) do not predict participants’ perception accuracy better than chance.


Alternative Hypothesis (H₁):
ERP and acoustic features do predict perception accuracy significantly better than chance.



# Table of Contents
- Hypotheses
- Data Preprocessing
- Feature Extraction
- Modeling & Classification
- Results & Interpretation



# 1. Data Preprocessing

## Behavioral Data Preprocessing




In [ ]:
# === 1. Parse Sentences_List1.txt ===
def parse_sentences_from_txt(txt_file):
    with open(txt_file, 'r', encoding='utf-8', errors='ignore') as f:
        lines = f.readlines()

    data = []
    for line in lines:
        line = line.strip()
        if line and line[0].isdigit() and '|' in line:
            try:
                idx, rest = line.split('.', 1)
                words = rest.strip().split()

                word1_raw = None
                function_word = None
                word2 = None

                for i, word in enumerate(words):
                    if '|' in word:
                        word1_raw = word
                        if i + 1 < len(words):
                            function_word = words[i + 1]
                        if i + 2 < len(words):
                            word2 = words[i + 2].replace('|', '')
                        break

                if word1_raw and function_word:
                    word1_clean = word1_raw.replace('|', '').lower()
                    function_word = function_word.lower()
                    full_sentence = ' '.join(w.replace('|', '') for w in words)

                    data.append({
                        'Index': int(idx),
                        'Word1': word1_clean,
                        'FunctionWord': function_word,
                        'Word2': word2.lower() if word2 else None,
                        'Full_Sentence': full_sentence.lower()
                    })

            except Exception as e:
                print(f"⚠️ Failed to parse line: {line}\n{e}")
                continue

    return pd.DataFrame(data)

# === 2. Check participant response logic ===
def check_sequence_after_word1(row, sentence_df):
    stim_text = str(row.get("StimTest.RESP", "")).lower().strip()
    stim_words = stim_text.split()

    idx = row.get("Index")
    match = sentence_df[sentence_df["Index"] == idx]
    if match.empty:
        return None

    word1 = match.iloc[0]["Word1"]
    func_word = match.iloc[0]["FunctionWord"]
    word2 = match.iloc[0]["Word2"]

    try:
        i = stim_words.index(word1)
    except ValueError:
        return None  # word1 not present → skip

    # case 1: Function word appears after word1
    if i + 1 < len(stim_words) and stim_words[i + 1] == func_word:
        return 0
    # case 2: word2 appears immediately after word1
    if i + 1 < len(stim_words) and stim_words[i + 1] == word2:
        return 1
    return None

# === 3. Try to read CSVs ===
def smart_read_csv(filepath):
    for enc in ['utf-8', 'utf-16', 'windows-1252']:
        for delim in [',', '\t', ';']:
            try:
                df = pd.read_csv(filepath, delimiter=delim, encoding=enc, engine='python', on_bad_lines='skip')
                if df.shape[1] >= 2:
                    return df
            except Exception:
                continue
    return None

# === 4. Main logic for printing ===
def print_evaluation_of_participants(root_dir, sentence_file):
    sentence_df = parse_sentences_from_txt(sentence_file)

    for participant_num in range(2, 28):  # 2 to 27 inclusive
        folder_path = os.path.join(root_dir, str(participant_num))
        if not os.path.isdir(folder_path):
            print(f"❌ Folder not found: {folder_path}")
            continue

        print(f"\n👤 Processing Participant {participant_num}")

        for fname in os.listdir(folder_path):
            if fname.lower().startswith("speechrate") and fname.lower().endswith(".csv"):
                input_path = os.path.join(folder_path, fname)

                df = smart_read_csv(input_path)
                if df is None:
                    print(f"❌ Could not read: {fname}")
                    continue

                df.rename(columns={"WavFiler": "Index", "CellT": "Condition"}, inplace=True)

                if not all(c in df.columns for c in ["Index", "Condition", "StimTest.RESP"]):
                    print(f"⚠️ Unexpected columns in {fname}")
                    continue

                df["Index"] = df["Index"].astype("Int64", errors="ignore")
                df = pd.merge(df, sentence_df, on="Index", how="left")

                df["Correct_Sequence (0/1)"] = df.apply(lambda row: check_sequence_after_word1(row, sentence_df), axis=1)
                df["Correct_Sequence (0/1)"] = df["Correct_Sequence (0/1)"].astype("Int64")
                df["Include_For_Stats"] = df["Correct_Sequence (0/1)"].isin([0, 1])

                # Print selected columns
                print(df[["Index", "Condition", "StimTest.RESP", "Correct_Sequence (0/1)", "Include_For_Stats"]].head(10))
                break

# === 5. Run ===
if __name__ == "__main__":
    root_dir = "/content/drive/MyDrive/2025_Summer (Distal Speech Rate)/Project_Code/Behavioral"
    sentence_file = "/content/drive/MyDrive/2025_Summer (Distal Speech Rate)/Project_Code/Stimuli/Sentences_List1.txt"
    print_evaluation_of_participants(root_dir, sentence_file)






| Value  | Meaning                                                         |
| ------ | --------------------------------------------------------------- |
| `0`    | **Correct** — The function word followed the expected `Word1` |
| `1`    | **Incorrect** — The function word **did not** follow `Word1`  |
| `<NA>` | **Not applicable** — One of the following:                   |
|        | - The response was too short                                    |
                    |
|        | - The sentence was malformed or incomplete                      |


Remove <NA> values and save the cleaned, annotated CSV file.










In [ ]:
import os
import pandas as pd
import traceback

def annotate_and_save_cleaned_participants(root_dir, sentence_file):
    print("🚀 Starting annotation process...")

    sentence_df = parse_sentences_from_txt(sentence_file)
    print(f"📘 Parsed {len(sentence_df)} sentences from sentence file.")

    for participant_num in range(2, 28):
        participant_folder = str(participant_num)
        folder_path = os.path.join(root_dir, participant_folder)

        print(f"\n📁 Checking folder: {folder_path}")

        if not os.path.isdir(folder_path):
            print(f"❌ Folder not found for participant {participant_num}")
            continue

        print(f"👤 Processing Participant {participant_num}")

        for fname in os.listdir(folder_path):
            print(f"🔍 Found file: {fname}")

            if fname.lower().startswith("speechrate") and fname.lower().endswith(".csv"):
                input_path = os.path.join(folder_path, fname)
                output_path = input_path.replace(".csv", "_cleaned_annotated.csv")

                print(f"📄 Processing file: {input_path}")

                try:
                    df = smart_read_csv(input_path)
                    if df is None or df.shape[1] < 2:
                        print(f"❌ Could not read valid table from: {fname}")
                        continue

                    df.rename(columns={"WavFiler": "Index", "CellT": "Condition"}, inplace=True)

                    expected_cols = ["Index", "Condition", "StimTest.RESP"]
                    if not all(col in df.columns for col in expected_cols):
                        print(f"⚠️ Unexpected columns in {fname}.")
                        print(f"   Found: {list(df.columns)}")
                        continue

                    df["Index"] = df["Index"].astype("Int64", errors="ignore")
                    df = pd.merge(df, sentence_df, on="Index", how="left")

                    df["Correct_Sequence (0/1)"] = df.apply(
                        lambda row: check_sequence_after_word1(row, sentence_df), axis=1)
                    df["Correct_Sequence (0/1)"] = df["Correct_Sequence (0/1)"].astype("Int64")
                    df["Include_For_Stats"] = df["Correct_Sequence (0/1)"].isin([0, 1])

                    df_cleaned = df[df["Include_For_Stats"]].copy()

                    if df_cleaned.empty:
                        print("⚠️ No usable rows after filtering. Skipping save.")
                    else:
                        df_cleaned.to_csv(output_path, index=False)
                        print(f"💾 Cleaned and saved to: {output_path}")
                        print(df_cleaned[["Index", "Condition", "StimTest.RESP", "Correct_Sequence (0/1)", "Include_For_Stats"]].head(10))

                except Exception as e:
                    print(f"❌ Error processing participant {participant_num}: {e}")
                    traceback.print_exc()

                break  # only process the first matching file

if __name__ == "__main__":
    print("✅ Script entry reached.")
    root_dir = "/content/drive/MyDrive/2025_Summer (Distal Speech Rate)/Project_Code/Behavioral"
    sentence_file = "/content/drive/MyDrive/2025_Summer (Distal Speech Rate)/Project_Code/Stimuli/Sentences_List1.txt"
    annotate_and_save_cleaned_participants(root_dir, sentence_file)



All Participants Summary:

In [ ]:

def combine_cleaned_csvs_to_summary(root_dir, output_file="All_Participants_Summary.csv"):
    all_rows = []

    for participant_num in range(2, 28):
        folder_path = os.path.join(root_dir, str(participant_num))
        if not os.path.isdir(folder_path):
            continue

        for fname in os.listdir(folder_path):
            if fname.endswith("_cleaned_annotated.csv"):
                csv_path = os.path.join(folder_path, fname)
                try:
                    df = pd.read_csv(csv_path)
                    df["Participant"] = participant_num  # Tag participant ID
                    all_rows.append(df)
                except Exception as e:
                    print(f"❌ Could not read {fname}: {e}")

    if all_rows:
        merged_df = pd.concat(all_rows, ignore_index=True)
        output_path = os.path.join(root_dir, output_file)

        # 💾 Save as CSV
        merged_df.to_csv(output_path, index=False)
        print(f"\n✅ Combined data saved to: {output_path}\n")

        # 🖨️ Show preview
        print(merged_df.head(10))
    else:
        print("⚠️ No cleaned annotated files found to combine.")

# ▶️ Run this block
if __name__ == "__main__":
    root_dir = "/content/drive/MyDrive/2025_Summer (Distal Speech Rate)/Project_Code/Behavioral"
    combine_cleaned_csvs_to_summary(root_dir)


## Load Onset Times



In [ ]:
csv_path = "/content/drive/MyDrive/2025_Summer (Distal Speech Rate)/Project_Code/Stimuli/SR_OnsetTimes_Final.csv"
df = pd.read_csv(csv_path)
print(df.head())



### Check If the Onset Times are in ms or seconds

In [ ]:


# === Set paths ===
audio_dir = "/content/drive/MyDrive/2025_Summer (Distal Speech Rate)/Project_Code/Stimuli"

# === Check a sample audio file ===
sample_file = "SR129.wav"
audio_path = os.path.join(audio_dir, sample_file)

# === Load audio and print duration ===
y, sr = librosa.load(audio_path, sr=16000)  # 16kHz
duration = len(y) / sr
print(f"\n🔊 '{sample_file}' duration: {duration:.2f} seconds")

# === Check if onset values are in ms or sec ===
onset_sample_ms = df.loc[0, "critical"]  # or any other column
onset_sample_sec = onset_sample_ms / 1000
print(f"Sample 'critical' onset (converted): {onset_sample_sec:.3f} sec")

# === Conclusion ===
if onset_sample_sec < duration:
    print("✅ Onset values are in milliseconds.")
else:
    print("⚠️ Onset values might already be in seconds — please double check.")



## Merge Behavioral & Timing Data

In [ ]:
import pandas as pd

# === 1. Load behavioral data ===
behavior_path = "/content/drive/MyDrive/2025_Summer (Distal Speech Rate)/Project_Code/Behavioral/All_Participants_Summary.csv"
df_behav = pd.read_csv(behavior_path)
print(f"✅ Loaded behavioral data: {df_behav.shape}")

# === 2. Load onset timing data ===
timing_path = "/content/drive/MyDrive/2025_Summer (Distal Speech Rate)/Project_Code/Stimuli/SR_OnsetTimes_Final.csv"
df_timing = pd.read_csv(timing_path)
print(f"✅ Loaded timing data: {df_timing.shape}")

# === 3. Rename columns for compatibility ===
df_timing.rename(columns={
    "sentence#": "Index",                    # Align with behavioral data
    "Word1": "Word1_Onset",
    "critical": "FunctionWord_Onset",
    "Word2": "Word2_Onset"
}, inplace=True)

# === 4. Merge on Index ===
df_merged = pd.merge(df_behav, df_timing, on="Index", how="inner")
print(f"🔁 Merged data shape: {df_merged.shape}")

# === 5. Drop any rows missing onset info ===
df_merged.dropna(subset=["Word1_Onset", "Word2_Onset", "FunctionWord_Onset"], inplace=True)
print(f"🧹 After dropna: {df_merged.shape}")

# === 6. Preview key columns ===
df_merged[["Index", "Word1_Onset", "FunctionWord_Onset", "Word2_Onset", "StimTest.RESP"]].head()


#2. Feature Engineering

### Feature Preprocessing

###Split the regions for each sentence

In [ ]:

# === Load and parse Sentences_List1.txt ===
def parse_sentence_components(txt_path):
    with open(txt_path, 'r', encoding='utf-8', errors='ignore') as f:
        lines = f.readlines()

    structured_data = []

    for line in lines:
        line = line.strip()
        if line and line[0].isdigit() and '|' in line:
            try:
                index_part, sentence = line.split('.', 1)
                index = int(index_part.strip())

                # Split the sentence into chunks using |
                chunks = sentence.strip().split('|')

                # Safety check
                if len(chunks) < 4:
                    print(f"⚠️ Skipping line {index}: not enough | segments.")
                    continue

                # Extract components
                preceding_context = chunks[0].strip()
                target_raw = chunks[1].strip()
                following_context = chunks[2].strip()
                extra_context = chunks[3].strip()

                # Target region is structured: Word1 + Function Word + start of Word2
                target_tokens = target_raw.split()
                if len(target_tokens) < 3:
                    print(f"⚠️ Skipping line {index}: target region has < 3 tokens.")
                    continue

                word1 = target_tokens[0]
                function_word = target_tokens[1]
                word2_start = target_tokens[2]

                target_region = f"{word1} {function_word} {word2_start}"

                structured_data.append({
                    "Index": index,
                    "Preceding_Context": preceding_context,
                    "Target_Region": target_region,
                    "Following_Context": following_context,
                    "Extra_Context": extra_context
                })

            except Exception as e:
                print(f"❌ Error parsing line: {line}\n{e}")
                continue

    return pd.DataFrame(structured_data)

# === Path to your sentence file ===
sentence_path = "/content/drive/MyDrive/2025_Summer (Distal Speech Rate)/Project_Code/Stimuli/Sentences_List1.txt"

# === Run the parser ===
df_sentences = parse_sentence_components(sentence_path)

# === Save parsed output ===
csv_path = "/content/drive/MyDrive/2025_Summer (Distal Speech Rate)/Project_Code/Sentences_List1_parsed.csv"
df_sentences.to_csv(csv_path, index=False)

# Optional: save as Pickle (faster for later use in Python)
pkl_path = "/content/drive/MyDrive/2025_Summer (Distal Speech Rate)/Project_Code/Sentences_List1_parsed.pkl"
df_sentences.to_pickle(pkl_path)

# === Display the first few rows ===
from IPython.display import display
display(df_sentences.head(10))






### Feature 1: Speech Rate of Preceding Context

In [ ]:

# === File paths ===
parsed_path = "/content/drive/MyDrive/2025_Summer (Distal Speech Rate)/Project_Code/Sentences_List1_parsed.csv"
onset_path = "/content/drive/MyDrive/2025_Summer (Distal Speech Rate)/Project_Code/Stimuli/SR_OnsetTimes_Final.csv"

# === Load parsed sentences ===
df_sentences = pd.read_csv(parsed_path)

# === Load onset times and rename columns for consistency ===
df_onsets = pd.read_csv(onset_path)
df_onsets.rename(columns={
    "sentence#": "Index",
    "critical": "Target_Onset_ms"
}, inplace=True)

# === Merge on Index ===
df = pd.merge(df_sentences, df_onsets, on="Index")

# === Convert onset time to seconds ===
df["Preceding_Duration_sec"] = df["Target_Onset_ms"] / 1000.0

# === Count syllables in Preceding_Context ===
dic = pyphen.Pyphen(lang='en')

def count_syllables(text):
    return sum(len(dic.inserted(word).split('-')) for word in text.split())

df["Preceding_Syllables"] = df["Preceding_Context"].apply(count_syllables)

# === Compute Distal Speech Rate ===
df["SpeechRate_Preceding"] = df["Preceding_Syllables"] / df["Preceding_Duration_sec"]

# === Display result ===
display(df[["Index", "Preceding_Context", "Preceding_Syllables", "Preceding_Duration_sec", "SpeechRate_Preceding"]].head(10))

# === Save final DataFrame to CSV ===
output_path = "/content/drive/MyDrive/2025_Summer (Distal Speech Rate)/Project_Code/Sentences_with_DistalSpeechRate.csv"
df.to_csv(output_path, index=False)
print(f"Data saved to: {output_path}")



###  Acoustic Features Predicting Perception


In [ ]:
# === New Section ===
# 📌 How well can acoustic features predict perception?

# 💡 We focus on two key acoustic features:
#    1. Speech rate of the preceding context (SpeechRate_Preceding)
#    2. Inter-word interval = Word2_Onset - Word1_Onset

# ✅ Step 0: Merge in distal speech rate data (if not yet merged)
df_rate = pd.read_csv("/content/drive/MyDrive/2025_Summer (Distal Speech Rate)/Project_Code/Sentences_with_DistalSpeechRate.csv")

# Only keep needed columns and merge on 'Index'
df_merged = df_merged.merge(df_rate[["Index", "SpeechRate_Preceding"]], on="Index", how="left")

# ✅ Step 1: Compute inter-word interval in seconds
df_merged["InterWordInterval"] = (df_merged["Word2_Onset"] - df_merged["Word1_Onset"]) / 1000.0

# ✅ Step 2: Select relevant features and drop missing
features_df = df_merged[[
    "SpeechRate_Preceding",
    "InterWordInterval",
    "Correct_Sequence (0/1)"
]].dropna()

# ✅ Step 3: Preview and optionally save
print("🔍 Feature sample:")
display(features_df.head())

# === Optional: Save features for modeling ===
features_df.to_csv("/content/drive/MyDrive/2025_Summer (Distal Speech Rate)/Project_Code/Acoustic_Features.csv", index=False)
print("💾 Saved feature data.")


#3. Inference:

## Accoustic Features Only

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix
import seaborn as sns
import matplotlib.pyplot as plt

# === Step 1: Prepare X and y ===
X = features_df[["SpeechRate_Preceding", "InterWordInterval"]]
y = features_df["Correct_Sequence (0/1)"]

# === Step 2: Train/test split ===
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# === Step 3: Fit logistic regression ===
model = LogisticRegression()
model.fit(X_train, y_train)

# === Step 4: Predict and evaluate ===
y_pred = model.predict(X_test)

print("📊 Classification Report:\n")
print(classification_report(y_test, y_pred))

# === Step 5: Confusion matrix ===
cm = confusion_matrix(y_test, y_pred)
sns.heatmap(cm, annot=True, fmt='d', cmap="Blues", xticklabels=["No", "Yes"], yticklabels=["No", "Yes"])
plt.xlabel("Predicted")
plt.ylabel("True")
plt.title("Confusion Matrix")
plt.show()

# === Step 6: Check coefficients ===
coef_df = pd.DataFrame({
    "Feature": X.columns,
    "Coefficient": model.coef_[0]
})
print("\n🧠 Feature Importance (Logistic Coefficients):")
display(coef_df)


Summary:



*   Acoustic features alone do contribute to perception, but not enough to reliably classify.
*   Faster preceding speech increases the chance that listeners perceive the function word.

*   Longer inter-word gaps slightly reduce perception probability.





# ERP

In [ ]:
import os
import mne
import numpy as np
import pandas as pd

# === Configuration ===
root_dir = '/content/drive/MyDrive/2025_Summer (Distal Speech Rate)/Project_Code/EEGData'
output_path = '/content/drive/MyDrive/2025_Summer (Distal Speech Rate)/Project_Code/ERP_Features.csv'

# Time windows (in seconds)
n1_window = (0.09, 0.13)     # ~100 ms
n280_window = (0.25, 0.32)   # ~280 ms

# Optional: specify channels to average over (e.g., central)
# If None, will average over all EEG channels
selected_channels = None  # e.g., ['Cz', 'FCz', 'CPz']

# === Processing loop ===
all_erp_data = []

subject_dirs = sorted([d for d in os.listdir(root_dir) if d.isdigit()], key=lambda x: int(x))

for subj in subject_dirs:
    clean_epo_path = os.path.join(root_dir, subj, f"{subj}_O1_epo_clean.fif")

    if not os.path.exists(clean_epo_path):
        print(f"❌ No cleaned epochs for subject {subj}, skipping.")
        continue

    print(f"📥 Processing subject {subj}...")

    try:
        epochs = mne.read_epochs(clean_epo_path, preload=True)
        times = epochs.times
        data = epochs.get_data()  # shape: (n_trials, n_channels, n_times)

        # Subset channels if specified
        if selected_channels:
            picks = mne.pick_channels(epochs.ch_names, selected_channels)
            data = data[:, picks, :]
        else:
            # Average over all EEG channels (not EOG or misc)
            picks = mne.pick_types(epochs.info, eeg=True)
            data = data[:, picks, :]

        # Get indices for N1 and N280 time windows
        n1_idx = np.where((times >= n1_window[0]) & (times <= n1_window[1]))[0]
        n280_idx = np.where((times >= n280_window[0]) & (times <= n280_window[1]))[0]

        # Extract mean amplitude per trial per window
        n1_amp = data[:, :, n1_idx].mean(axis=(1, 2))
        n280_amp = data[:, :, n280_idx].mean(axis=(1, 2))
        print(f"Size n1_amp: {n1_amp.shape}.\n Size n280_amp: {n280_amp.shape}.")

        # Trial-level dataframe
        n_trials = len(n1_amp)
        df = pd.DataFrame({
            'Participant': [int(subj)] * n_trials,
            'Index': np.arange(n_trials),
            'N1_amplitude': n1_amp,
            'N280_amplitude': n280_amp
        })
        all_erp_data.append(df)

    except Exception as e:
        print(f"⚠️ Error processing subject {subj}: {e}")
        continue

# === Combine and save ===
erp_df = pd.concat(all_erp_data, ignore_index=True)
erp_df.to_csv(output_path, index=False)

print(f"\n✅ Saved ERP feature file: {output_path}")
display(erp_df.head())


#ERP + 2 Features

 ### Merge ERP + Acoustic Data

In [ ]:
from logging import critical
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns

# === Step 1: Build acoustic feature set ===
# Make sure df_merged includes these columns!
features_df = df_merged[[
    "Participant",                # From df_merged
    "Index",                      # Trial-level ID
    "SpeechRate_Preceding",
    "InterWordInterval",
    "Correct_Sequence (0/1)"
]].dropna()

print("✅ Acoustic features:")
display(features_df.head())

# === Step 2: Load ERP features ===
erp_df = pd.read_csv('/content/drive/MyDrive/2025_Summer (Distal Speech Rate)/Project_Code/ERP_Features.csv')

print("✅ ERP features:")
display(erp_df.head(70))

# === Step 3: Merge acoustic + ERP data ===
combined_df = features_df.merge(erp_df, on=["Participant", "Index"], how="inner")
combined_df = combined_df.dropna()

print(f"✅ Combined dataset shape: {combined_df.shape}")
display(combined_df.head(20))
# InterWordInterval - word2 - critical region - it will be then the same for both versions and its more direvtly mesearures the time between 1 onset and then next onset

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
# same number of predictors , as many obsevration for each predictors  ( we need to have the same erp amplituede and same sentence)
# === Step 1: Define predictors and target ===
X = combined_df[[
    "SpeechRate_Preceding",
    "InterWordInterval",
    "N1_amplitude",
    "N280_amplitude"
]]
y = combined_df["Correct_Sequence (0/1)"]

# === Step 2: Train/test split ===
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# === Step 3: Fit logistic regression ===
model = LogisticRegression(max_iter=1000)
model.fit(X_train, y_train)

# === Step 4: Predict and evaluate ===
y_pred = model.predict(X_test)

print("📊 Classification Report:")
print(classification_report(y_test, y_pred))

# === Step 5: Confusion matrix ===
cm = confusion_matrix(y_test, y_pred)
sns.heatmap(cm, annot=True, fmt='d', cmap="Blues", xticklabels=["Not Perceived", "Perceived"], yticklabels=["Not Perceived", "Perceived"])
plt.xlabel("Predicted")
plt.ylabel("True")
plt.title("Confusion Matrix")
plt.show()

# === Step 6: Feature importances ===
coef_df = pd.DataFrame({
    "Feature": X.columns,
    "Coefficient": model.coef_[0]
})
print("🧠 Feature Importances:")
display(coef_df)


| Feature                   | Coefficient | Interpretation                                                |
| ------------------------- | ----------- | ------------------------------------------------------------- |
| **SpeechRate\_Preceding** | `+0.1138`   | ⬆️ Faster preceding speech increases likelihood of perception |
| **InterWordInterval**     | `–0.0406`   | ⬇️ Longer gap between words slightly reduces perception       |
| **N1\_amplitude**         | `–4.19e-6`  | ⬇️ More negative N1 is weakly associated with not perceiving  |
| **N280\_amplitude**       | `+1.83e-8`  | ⬆️ Positive N280 might weakly support perception              |


In [ ]:
from xgboost import XGBClassifier

# === Step 3: Fit XGBoost ===
xgb_model = XGBClassifier(use_label_encoder=False, eval_metric='logloss', random_state=42)
xgb_model.fit(X_train, y_train)

# === Step 4: Predict and evaluate ===
y_pred = xgb_model.predict(X_test)

print("📊 Classification Report:")
print(classification_report(y_test, y_pred))

# === Step 5: Confusion matrix ===
cm = confusion_matrix(y_test, y_pred)
sns.heatmap(cm, annot=True, fmt='d', cmap="Blues", xticklabels=["Not Perceived", "Perceived"], yticklabels=["Not Perceived", "Perceived"])
plt.xlabel("Predicted")
plt.ylabel("True")
plt.title("Confusion Matrix")
plt.show()




### Add Additional Features

In [ ]:
import re

# Load acoustic summary
acoustic_df = pd.read_csv("/content/drive/MyDrive/2025_Summer (Distal Speech Rate)/Project_Code/Extracted_Features/summary_acoustic_features.csv")

# Remove extension and extract numeric part (e.g., from 'SR105.wav' get '105')
acoustic_df["file_id"] = acoustic_df["file"].str.replace(".wav", "", regex=False)
acoustic_df["Index"] = acoustic_df["file_id"].apply(lambda x: int(re.search(r"\d+", x).group()))

# Show result
acoustic_df.head()



In [ ]:
# Make sure Index is of same type in both DataFrames
combined_df["Index"] = combined_df["Index"].astype(int)
acoustic_df["Index"] = acoustic_df["Index"].astype(int)

# Drop duplicate filename columns before merging
acoustic_df_clean = acoustic_df.drop(columns=["file", "file_id"])

# ✅ Merge on Index
df_all = combined_df.merge(acoustic_df_clean, on="Index", how="left")

# ✅ Check the final dataset
print("✅ Final shape with all features:", df_all.shape)
display(df_all.head())


## Inference ERP + Additional Features

## Logistic Regression

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix
import seaborn as sns
import matplotlib.pyplot as plt

# 🎯 Define target variable
target = "Correct_Sequence (0/1)"

# 🧠 Define feature columns (exclude Participant, Index, and target)
exclude_cols = ["Participant", "Index", target]
feature_cols = [col for col in df_all.columns if col not in exclude_cols]

# 🧹 Drop any rows with missing values
df_model = df_all[feature_cols + [target]].dropna()

# 🪄 Split into train/test
X = df_model[feature_cols]
y = df_model[target]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# 🤖 Train logistic regression
model = LogisticRegression(max_iter=1000)
model.fit(X_train, y_train)

# 📊 Evaluate
y_pred = model.predict(X_test)
print("📋 Classification Report:")
print(classification_report(y_test, y_pred))

# 📉 Confusion matrix
cm = confusion_matrix(y_test, y_pred)
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=["Pred 0", "Pred 1"], yticklabels=["True 0", "True 1"])
plt.title("Confusion Matrix")
plt.xlabel("Predicted")
plt.ylabel("True")
plt.show()


## XGBoost Classifier

In [ ]:
# ✅ Set the final merged dataset
df_final = df_all  # Your full dataset with acoustic + ERP + MFCC features

# === Step 1: Define features and target ===
X = df_final.drop(columns=["Correct_Sequence (0/1)", "Participant", "Index"], errors="ignore")
y = df_final["Correct_Sequence (0/1)"]

# === Step 2: Train-test split ===
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42, stratify=y)

# === Step 3: Scale features ===
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# === Step 4: Train XGBoost classifier ===
from xgboost import XGBClassifier
xgb = XGBClassifier(use_label_encoder=False, eval_metric='logloss', random_state=42)
xgb.fit(X_train_scaled, y_train)

# === Step 5: Evaluate model ===
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay
y_pred = xgb.predict(X_test_scaled)

print("\n📊 Classification Report:")
print(classification_report(y_test, y_pred))

# === Confusion Matrix ===
cm = confusion_matrix(y_test, y_pred)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=["Not Perceived", "Perceived"])
disp.plot(cmap="Blues")



Yes — acoustic features and ERP signals together can reliably predict perceptual outcomes.

The model successfully distinguishes perceived from not perceived items with 96% accuracy.

This supports the hypothesis that both neural (ERP) and speech timing/acoustic cues carry meaningful information about perception.

### CROSS VALIDATION

In [ ]:
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import classification_report, confusion_matrix
import xgboost as xgb
import numpy as np

# Features and target
X = df_all.drop(columns=["Correct_Sequence (0/1)", "Participant", "Index"], errors="ignore")
y = df_all["Correct_Sequence (0/1)"]

# Prepare cross-validation
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
fold = 1
scores = []

for train_index, test_index in skf.split(X, y):
    X_train, X_test = X.iloc[train_index], X.iloc[test_index]
    y_train, y_test = y.iloc[train_index], y.iloc[test_index]

    model = xgb.XGBClassifier(use_label_encoder=False, eval_metric="logloss")
    model.fit(X_train, y_train)

    y_pred = model.predict(X_test)

    print(f"\n📂 Fold {fold} Classification Report:")
    print(classification_report(y_test, y_pred))
    print("Confusion Matrix:")
    print(confusion_matrix(y_test, y_pred))

    scores.append(model.score(X_test, y_test))
    fold += 1

print("\n✅ Average Accuracy Across Folds:", np.mean(scores))


Conlusion:

The model exhibited low variance across confusion matrices during stratified K-fold cross-validation, meaning it consistently made accurate predictions on both classes with minimal fluctuations between folds. This stability suggests that the model has not overfit to any specific subset of the data, and instead has learned patterns that generalize well across the entire dataset.

## Feature Importance

In [ ]:
#Feature importance
import matplotlib.pyplot as plt
import pandas as pd

importance_scores = xgb.feature_importances_
feature_names = X.columns

importance_df = pd.DataFrame({
    "Feature": feature_names,
    "Importance": importance_scores
}).sort_values(by="Importance", ascending=False)

# Show top 10 features
print("\n🔝 Top 10 Most Important Features:")
print(importance_df.head(10))

# Plot top 15
plt.figure(figsize=(10, 6))
plt.barh(importance_df["Feature"][:15][::-1], importance_df["Importance"][:15][::-1])
plt.xlabel("Importance Score")
plt.title("Top 15 Most Important Features")
plt.tight_layout()
plt.show()


# mTRF

In [ ]:
# === Import Required Libraries ===
import os
import numpy as np
import pandas as pd
import mne
from sklearn.linear_model import Ridge
from sklearn.metrics import r2_score
from tqdm import tqdm

# === Set Paths ===
eeg_dir = "/content/drive/MyDrive/2025_Summer (Distal Speech Rate)/Project_Code/EEGData"
acoustic_dir = "/content/drive/MyDrive/2025_Summer (Distal Speech Rate)/Project_Code/Extracted_Features"

# === Run mTRF for All Participants ===
def run_mtrf_all_participants(eeg_dir, acoustic_dir):
    global_r2 = []

    # Get available acoustic files
    acoustic_files = sorted([
        f for f in os.listdir(acoustic_dir)
        if f.endswith("_temporal.csv") and f.startswith("SR")
    ])
    acoustic_base = [f.replace("_temporal.csv", "") for f in acoustic_files]
    print(f"🔍 Found {len(acoustic_files)} acoustic files.")

    participants = sorted([d for d in os.listdir(eeg_dir) if d.isdigit()])
    print(f"👥 Participants found: {participants}")

    for pid in participants:
        print(f"\n🧠 Running mTRF for Participant {pid}")
        eeg_path = os.path.join(eeg_dir, pid, f"{pid}_O1_epo_clean.fif")

        if not os.path.exists(eeg_path):
            print(f"❌ EEG file not found for {pid}")
            continue

        try:
            # Load EEG epochs
            epochs = mne.read_epochs(eeg_path, preload=True)
            eeg_data = epochs.get_data()  # [n_epochs, n_channels, n_times]
            trial_eeg = [trial.T for trial in eeg_data]  # List of [time x channels]

            # Match acoustic files for participant (e.g., SR53_N, SR53_E)
            matching_files = [
                f for f in acoustic_base if f.split("_")[0].endswith(pid)
            ]
            trial_audio = []

            for match in matching_files:
                path = os.path.join(acoustic_dir, match + "_temporal.csv")
                if os.path.exists(path):
                    df = pd.read_csv(path)
                    trial_audio.append(df.values)
                else:
                    print(f"⚠️ Missing acoustic file: {path}")

            # Align and clean mismatched trials
            n_trials = min(len(trial_audio), len(trial_eeg))
            if n_trials < 1:
                print(f"⚠️ Too few matching trials ({n_trials}) — skipping {pid}")
                continue

            trial_audio = trial_audio[:n_trials]
            trial_eeg = trial_eeg[:n_trials]

            # === Run mTRF Encoding: Acoustic → EEG ===
            r2_scores = []
            for i in range(n_trials):
                X = trial_audio[i]  # acoustic features: time x features
                Y = trial_eeg[i]    # EEG: time x channels

                if X.shape[0] != Y.shape[0]:
                    min_len = min(X.shape[0], Y.shape[0])
                    X, Y = X[:min_len], Y[:min_len]

                model = Ridge(alpha=1.0)
                model.fit(X, Y)
                Y_pred = model.predict(X)
                r2 = r2_score(Y, Y_pred, multioutput="uniform_average")
                r2_scores.append(r2)

            mean_r2 = np.mean(r2_scores)
            print(f"📊 Participant {pid} — Mean R²: {mean_r2:.4f}")
            global_r2.append((pid, mean_r2))

        except Exception as e:
            print(f"⚠️ Error with participant {pid}: {e}")
            continue

    return global_r2


# === Run the mTRF Pipeline ===
results = run_mtrf_all_participants(eeg_dir, acoustic_dir)

# === Display Final Results ===
df_r2 = pd.DataFrame(results, columns=["Participant", "Mean_R2"])
print("\n🎯 Final mTRF Results Summary:")
display(df_r2)


Final mTRF Results Interpretation:

| Participant | Mean R² | Interpretation                                                                                                                   |
| ----------- | ------- | -------------------------------------------------------------------------------------------------------------------------------- |
| **10**      | 0.672   | ✅ **Strong fit** — EEG responses are well predicted by acoustic features. Likely good data quality and strong cortical tracking. |
| **11**      | 0.591   | ✅ **Moderate to strong fit** — solid alignment between features and EEG.                                                         |
| **12**      | 0.434   | ⚠️ **Moderate fit** — acoustic features explain some EEG variance, but less than others.                                         |
| **13**      | 0.752   | 🌟 **Excellent fit** — top performer, suggesting highly reliable EEG tracking of acoustic input.                                 |
| **14**      | 0.707   | ✅ **Strong fit** — robust EEG prediction, clear feature-driven responses.                                                        |
| **15**      | 0.670   | ✅ **Strong fit** — similar to P10 and P14, indicates meaningful encoding of acoustic features.                                   |
| **17**      | 0.388   | ⚠️ **Moderate to weak fit** — model performance is lower, could reflect noisier EEG or weaker entrainment.                       |
| **18**      | 0.533   | ✅ **Moderate fit** — some variability explained by features.                                                                     |
| **20**      | 0.323   | ⚠️ **Weak fit** — limited prediction of EEG from features; may be noise or poor alignment.                                       |
| **21**      | 0.541   | ✅ **Moderate fit** — over half of EEG variance explained, decent model performance.                                              |
| **22**      | 0.662   | ✅ **Strong fit** — EEG tracks acoustic structure well.                                                                           |
| **25**      | 0.523   | ✅ **Moderate fit** — consistent with others, decent prediction.                                                                  |
| **26**      | 0.595   | ✅ **Moderate to strong fit** — good model performance.                                                                           |
| **27**      | 0.726   | 🌟 **Excellent fit** — one of the best fits, highly feature-driven EEG.                                                          |
| **3**       | 0.538   | ✅ **Moderate fit** — typical tracking strength.                                                                                  |
| **6**       | 0.538   | ✅ **Moderate fit** — similar to P3.                                                                                              |
| **7**       | 0.557   | ✅ **Moderate fit** — average EEG-feature alignment.                                                                              |
| **8**       | 0.418   | ⚠️ **Moderate to weak fit** — below-average R²; possibly noisier EEG or timing issues.                                           |
